In [14]:
import os
import pandas as pd
import torch
import numpy as np

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")
DATA_DIR = "/kaggle/input/datasets/williamalxndr/chess-dataset" if IS_KAGGLE else "."

states      = torch.load(os.path.join(DATA_DIR, "states.pt"))
legal_masks = torch.load(os.path.join(DATA_DIR, "legal_masks.pt"))
target      = pd.read_csv(os.path.join(DATA_DIR, "target.csv"))

In [15]:
import sys
import os
import subprocess

if IS_KAGGLE:
    repo_dir = "/kaggle/working/mcts-engine"
    if not os.path.exists(repo_dir):
        subprocess.run(
            ["git", "clone", "https://github.com/williamalxndr/mcts-engine.git", repo_dir],
            check=True,
        )
    os.chdir(repo_dir)
    subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)
    sys.path.append('.')
else:
    sys.path.append('..')

In [16]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [17]:
row = states.shape[0]

print(states.shape)
print(legal_masks.shape)

torch.Size([707542, 30, 8, 8])
torch.Size([707542, 4672])


In [19]:
policy = torch.tensor(target.policy.values).long()
value = torch.tensor(target.value.values).float()


In [20]:
from core import factory

network = factory.build_network("chess")

In [21]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [22]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states))
assert len(states) == len(policy) == len(value) == len(legal_masks) 

train_states, test_states = states[:TRAIN_SIZE],      states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE],       policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],        value[TRAIN_SIZE:]
train_masks,  test_masks  = legal_masks[:TRAIN_SIZE],  legal_masks[TRAIN_SIZE:]

assert len(train_states) == len(train_policy) == len(train_value) == len(train_masks)
assert len(test_states)  == len(test_policy)  == len(test_value)  == len(test_masks)

In [23]:
from torch.nn.parallel import DataParallel

def save_network(network, game: str = "chess", version: str = "v2", file_name: str = "example", parent_dir: str = "checkpoints", path: str = None):
    real_network = network.module if isinstance(network, DataParallel) else network
    real_network.save(game=game, version=version, file_name=file_name, parent_dir=parent_dir, path=path)

In [ ]:
import numpy as np

mask_count    = test_masks[:4096].sum(dim=1).float()
channel_count = test_states[:4096, 20, 0, 0] * 218   # ChessEncoderV2 ch.20 = legal_moves/218, independent source

avg_legal_mask    = mask_count.mean().item()
avg_legal_channel = channel_count.mean().item()

print("avg legal moves (from mask)   :", avg_legal_mask)
print("avg legal moves (from ch. 20) :", avg_legal_channel)
print("max |diff| across samples     :", (mask_count - channel_count).abs().max().item())

print("uniform-over-legal baseline (mask)   :", np.log(avg_legal_mask))
print("uniform-over-legal baseline (ch. 20) :", np.log(avg_legal_channel))

In [24]:
import time
import torch
import numpy as np
from torch import optim
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import DataParallel
import itertools

from core.network import PolicyValueNetwork


def evaluate(network, test_states, test_policy, test_value, test_masks,
             policy_loss_fn, value_loss_fn, batch_size=256, eval_samples=4096):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    n = min(len(test_states), eval_samples)

    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]
            batch_mask   = test_masks[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_mask.to(policy_head.device), float("-inf"))

            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()

    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          train_masks: torch.Tensor, 
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          test_masks: torch.Tensor | None = None,
          batch_size: int = 256,
          num_epoch: int | None = None,
          duration_hour: float | None = None,
          save_path: str | None = "default.pt"):

    if num_epoch is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_epoch or duration_hour")

    start = time.time()
    step = 0
    best_val_loss = float('inf') # Initialize best validation loss tracking

    # Distributed training
    if torch.cuda.is_available():
        network = DataParallel(network)

    # Dataset
    dataset = TensorDataset(train_states, train_policy, train_value.unsqueeze(-1), train_masks)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    total_steps = len(dataloader) * num_epoch if num_epoch is not None else None

    print(len(dataloader))

    warmup_steps = len(dataloader) // 10
    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps if total_steps else len(dataloader) * duration_hour * 2)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        eval_n = min(len(test_states), 4096)
        test_states = test_states[:eval_n].to(device=DEVICE)
        test_policy = test_policy[:eval_n].to(device=DEVICE)
        test_value  = test_value[:eval_n].unsqueeze(-1).to(device=DEVICE)
        test_masks  = test_masks[:eval_n]

    epoch_iter = range(num_epoch) if num_epoch is not None else itertools.count()

    save_network(network, path=save_path)
    for _ in epoch_iter:
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        for batch_states, batch_policy, batch_value, batch_masks in dataloader:
            batch_states = batch_states.to(DEVICE)
            batch_policy = batch_policy.to(DEVICE)
            batch_value  = batch_value.to(DEVICE)
            batch_masks  = batch_masks.to(DEVICE)

            optimizer.zero_grad()
            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_masks, float("-inf"))

            policy_loss = policy_loss_fn(policy_head, batch_policy)
            value_loss  = value_loss_fn(value_head, batch_value)
            loss = policy_loss + value_loss
            loss.backward()

            optimizer.step()
            scheduler.step()

            if step % 1 == 0:
                elapsed = time.time() - start
                print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={(value_loss.item()):.4f} | lr: {scheduler.get_last_lr()[0]:.8f} | {elapsed:.0f}s")

            step += 1

        if has_test:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, test_masks,
                policy_loss_fn, value_loss_fn
            )
            
            current_val_loss = val_policy_loss + val_value_loss
            
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f} | total_val={current_val_loss:.4f}")

            # Check if this is the best model we've seen so far
            if save_path is not None and current_val_loss < best_val_loss:
                print(f"    [save] Validation loss improved from {best_val_loss:.4f} to {current_val_loss:.4f}. Saving model to '{save_path}'...")
                best_val_loss = current_val_loss
                save_network(network, path=save_path)


In [26]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    train_masks=train_masks,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    test_masks=test_masks,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=512,
)

1244
Network saved at default.pt
[0] loss=7.0729 | policy=5.3769 | value=1.6960 | lr: 0.00000540 | 4s
[1] loss=6.7361 | policy=5.0418 | value=1.6943 | lr: 0.00000779 | 13s
[2] loss=6.9959 | policy=5.2959 | value=1.7000 | lr: 0.00001019 | 19s
[3] loss=6.8377 | policy=5.0642 | value=1.7734 | lr: 0.00001258 | 24s
[4] loss=6.5670 | policy=4.8516 | value=1.7154 | lr: 0.00001498 | 31s
[5] loss=6.5227 | policy=4.8159 | value=1.7068 | lr: 0.00001737 | 40s
[6] loss=6.5178 | policy=4.7720 | value=1.7458 | lr: 0.00001977 | 47s


KeyboardInterrupt: 